# Teacher Preview on GPU

Run a larger teacher preview on a rented GPU using batched inference. The notebook supports vLLM for high GPU utilization and Hugging Face Transformers as a fallback.


## 1. Configure Run

Set `MODEL_NAME` to a Hugging Face repo id or a local model path on the GPU machine.

In [ ]:
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
BACKEND = "vllm"  # "vllm" or "hf"

NUM_SAMPLES = 100
BATCH_SIZE = 16
MAX_NEW_TOKENS = 1024
MAX_RETRIES = 2
TEMPERATURE = 0.0

# vLLM settings
TENSOR_PARALLEL_SIZE = 1
GPU_MEMORY_UTILIZATION = 0.9
MAX_MODEL_LEN = None

INPUT_PATH = "data/gsm8k_clean_train.jsonl"
OUTPUT_PATH = "data/gsm8k_teacher_preview_100.jsonl"


## 2. Imports

In [ ]:
import json
import sys
from pathlib import Path

from tqdm.auto import tqdm

candidate_roots = [
    Path.cwd(),
    Path.cwd() / "strategy-distill-rl",
    Path("/content/strategy-distill-rl"),
]
PROJECT_ROOT = next(
    root for root in candidate_roots
    if (root / "scripts/generate_teacher_preview.py").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

from scripts.generate_teacher_preview import (
    build_retry_prompt,
    build_teacher_record,
    read_jsonl,
    write_jsonl,
)
from src.teacher.local_teacher import (
    generate_teacher_outputs_hf,
    generate_teacher_outputs_vllm,
    load_local_teacher,
    load_vllm_teacher,
)
from src.utils.prompts import build_strategy_teacher_prompt


## 3. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## 4. Load Data and Model

In [ ]:
input_path = PROJECT_ROOT / INPUT_PATH

if not input_path.exists():
    print(f"Missing {input_path}. Preparing GSM8K clean JSONL files first...")
    from src.data.clean_gsm8k import clean_split
    from src.data.load_gsm8k import load_gsm8k

    dataset = load_gsm8k()
    clean_split(dataset["train"], PROJECT_ROOT / "data/gsm8k_clean_train.jsonl")
    clean_split(dataset["test"], PROJECT_ROOT / "data/gsm8k_clean_test.jsonl")

examples = read_jsonl(input_path, limit=NUM_SAMPLES)
print(f"Loaded {len(examples)} examples")

if BACKEND == "vllm":
    tokenizer, llm = load_vllm_teacher(
        MODEL_NAME,
        tensor_parallel_size=TENSOR_PARALLEL_SIZE,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        max_model_len=MAX_MODEL_LEN,
    )
    model = None
elif BACKEND == "hf":
    tokenizer, model = load_local_teacher(MODEL_NAME)
    llm = None
else:
    raise ValueError(f"Unknown BACKEND: {BACKEND}")

print(f"Teacher loaded with backend={BACKEND}")


## 5. Generate Teacher Preview

Batch inference improves GPU utilization because the model processes many prompts in one forward pass instead of launching tiny one-sample workloads. Failed samples are retried in smaller pending batches, and JSONL is written after each batch.


In [ ]:
def batched(items, batch_size):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def generate_batch(prompts):
    if BACKEND == "vllm":
        return generate_teacher_outputs_vllm(
            tokenizer=tokenizer,
            llm=llm,
            prompts=prompts,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
        )

    return generate_teacher_outputs_hf(
        tokenizer=tokenizer,
        model=model,
        prompts=prompts,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
    )


output_path = PROJECT_ROOT / OUTPUT_PATH
teacher_records = []
example_batches = list(batched(examples, BATCH_SIZE))

for batch_examples in tqdm(example_batches, desc="Generating teacher traces"):
    pending = [
        {
            "example": example,
            "prompt": build_strategy_teacher_prompt(example["question"]),
        }
        for example in batch_examples
    ]
    batch_records = []

    for attempt in range(MAX_RETRIES + 1):
        if not pending:
            break

        prompts = [item["prompt"] for item in pending]
        raw_outputs = generate_batch(prompts)
        next_pending = []

        for item, raw_output in zip(pending, raw_outputs):
            example = item["example"]
            record = build_teacher_record(example, raw_output)

            if record["is_usable"] == 1:
                batch_records.append(record)
            elif attempt < MAX_RETRIES:
                next_pending.append({
                    "example": example,
                    "prompt": build_retry_prompt(example["question"]),
                })

        pending = next_pending

    # Keep only usable records for the teacher dataset.
    teacher_records.extend(batch_records)
    write_jsonl(teacher_records, output_path)

    print(f"Saved usable records so far: {len(teacher_records)} -> {output_path}")

print(f"Done. Saved {len(teacher_records)} usable records to {output_path}")


## 6. Summary

In [ ]:
rows = [json.loads(line) for line in output_path.open("r", encoding="utf-8")]
total = len(rows)
correct = sum(row["is_correct"] for row in rows)
format_valid = sum(row["is_format_valid"] for row in rows)
usable = sum(row["is_usable"] for row in rows)

print(f"Total saved usable rows: {total}")
if total:
    print(f"Correct: {correct}/{total} = {correct / total:.1%}")
    print(f"Format valid: {format_valid}/{total} = {format_valid / total:.1%}")
    print(f"Usable: {usable}/{total} = {usable / total:.1%}")

failed_ids = [row["id"] for row in rows if not row["is_usable"]]
print("Failed usable ids in saved file:", failed_ids[:50])


## 7. Inspect Failures

In [ ]:
for row in rows:
    if row["is_usable"]:
        continue

    print("=" * 100)
    print("id:", row["id"])
    print("ground_truth:", row["ground_truth"])
    print("teacher_answer:", row["teacher_answer"])
    print("is_correct:", row["is_correct"])
    print("is_format_valid:", row["is_format_valid"])
    print("format_checks:", row["format_checks"])
    print("raw_teacher_output preview:")
    print((row["raw_teacher_output"] or "")[:1200])